# Problem 31: Multi-Tool Conflict Resolution

**Difficulty:** Hard | **Framework:** CrewAI

**Categories:** tool-calling

This notebook accompanies [Agent Foundry](https://agent-foundry-theta.vercel.app/problems/multi-tool-conflict-resolution).

## 1. Install Dependencies

In [ ]:
!pip install -q crewai crewai-tools

## 2. Set Up Your API Key

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. Constraints

- The agent must call both data sources before answering
- When data conflicts, the agent must acknowledge the discrepancy
- The agent must not silently pick one source over the other
- The resolution strategy must be transparent to the user


## 4. Starter Code (has a bug — fix it!)

The code below has an intentional issue. Read the problem description and fix it.

In [ ]:
from crewai import Agent, Task, Crew
from crewai.tools import tool

@tool
def get_price_source_a(product: str) -> str:
    """Get product price from Source A (retail aggregator)."""
    prices = {
        "laptop x1": {"price": "$999", "in_stock": True, "rating": "4.2/5"},
        "headphones z3": {"price": "$199", "in_stock": True, "rating": "4.7/5"},
        "keyboard k5": {"price": "$149", "in_stock": False, "rating": "4.0/5"},
    }
    data = prices.get(product.lower())
    if data:
        return f"Source A - {product}: Price {data['price']}, In Stock: {data['in_stock']}, Rating: {data['rating']}"
    return f"Source A: Product '{product}' not found"

@tool
def get_price_source_b(product: str) -> str:
    """Get product price from Source B (manufacturer database)."""
    prices = {
        "laptop x1": {"price": "$1,049", "in_stock": True, "rating": "4.5/5"},
        "headphones z3": {"price": "$199", "in_stock": False, "rating": "4.6/5"},
        "keyboard k5": {"price": "$129", "in_stock": True, "rating": "4.1/5"},
    }
    data = prices.get(product.lower())
    if data:
        return f"Source B - {product}: Price {data['price']}, In Stock: {data['in_stock']}, Rating: {data['rating']}"
    return f"Source B: Product '{product}' not found"

# BUG: The agent often calls only one source, or calls both but silently picks
# whichever result it saw first — even when the two sources disagree on price or stock
# TODO: Make the agent call both sources and explicitly handle conflicts
assistant = Agent(
    role="Shopping Assistant",
    goal="Help users find product information",
    backstory="You are a shopping assistant. Look up product information for users.",
    tools=[get_price_source_a, get_price_source_b],
)

task = Task(
    description="What's the price of Laptop X1 and is it in stock?",
    expected_output="Product information with pricing and availability",
    agent=assistant,
)

crew = Crew(agents=[assistant], tasks=[task])
print(crew.kickoff())

## 5. Your Solution

Modify the code above or write your fixed version below.

In [ ]:
# Write your solution here


## 6. Hints

1. Update the system prompt to instruct the agent to always cross-reference both sources and flag differences
2. Consider adding a reconciliation step that compares outputs and highlights conflicts before forming a final answer
3. You can implement a scoring or confidence-based strategy where the agent explains which source it trusts more and why


## 7. Evaluation Criteria

Check your solution against these criteria:

- Both sources are consulted
- Conflicts are flagged
- No silent preference
- Resolution is transparent
